# Phase 3 — Model Training Pipeline
### Stock Sentiment Analyzer · Direction Classification

**Task:** Predict next-day price direction as a 3-class problem:
- `1` → Up (close > +0.5%)
- `-1` → Down (close < -0.5%)
- `0` → Flat

**Model stack:**
| Layer | Model | Purpose |
|---|---|---|
| Baseline | Logistic Regression | Benchmark |
| Primary | XGBoost (GPU) | Champion candidate |

**Key ML discipline:**
- Temporal train/test split — no shuffling, no leakage
- `TimeSeriesSplit` cross-validation
- SHAP feature explainability
- MLflow experiment tracking
- Champion model serialised to `.pkl`

## 0 · Setup & Installs
All core packages are pre-installed on Kaggle. We only need `shap` and `mlflow`.

In [ ]:
!pip install -q shap mlflow

In [ ]:
import os, json, pickle, warnings, shutil
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import shap
import mlflow

from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import TimeSeriesSplit, cross_validate
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score,
    recall_score, confusion_matrix, classification_report
)
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

warnings.filterwarnings("ignore")
print("✅ Imports OK")

## 1 · Config & Paths

> **Kaggle tip:** Upload `feature_matrix.csv` via *Add Data → Upload* and update `DATA_PATH` to match your dataset path (e.g. `/kaggle/input/your-dataset-name/feature_matrix.csv`).

In [ ]:
# ── Update this path to your uploaded dataset ──────────────────────
DATA_PATH = "/kaggle/input/stock-sentiment-features/feature_matrix.csv"

# ── Output dirs (Kaggle working dir is /kaggle/working) ────────────
OUT_DIR   = Path("/kaggle/working/phase3")
MODEL_DIR = OUT_DIR / "models"
PLOT_DIR  = OUT_DIR / "plots"
for d in [MODEL_DIR, PLOT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

REGISTRY  = OUT_DIR / "runs.json"
TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")

# ── ML config ──────────────────────────────────────────────────────
TARGET    = "price_direction"
DROP_COLS = ["ticker", "date", TARGET]
N_SPLITS  = 3
LABEL_MAP = {-1: "Down", 0: "Flat", 1: "Up"}

print(f"Output dir : {OUT_DIR}")
print(f"Timestamp  : {TIMESTAMP}")

## 2 · Data Loading & Preprocessing

In [ ]:
df = pd.read_csv(DATA_PATH)
print(f"Loaded {len(df)} rows × {len(df.columns)} columns")
print(f"Tickers   : {sorted(df['ticker'].unique())}")
print(f"Date range: {df['date'].min()} → {df['date'].max()}")
print(f"\nTarget distribution:")
print(df[TARGET].value_counts().rename(index=LABEL_MAP).to_string())

In [ ]:
# Sort chronologically — CRITICAL for time-series integrity
df["date_parsed"] = pd.to_datetime(df["date"], dayfirst=True, errors="coerce")
df = df.sort_values(["ticker", "date_parsed"]).reset_index(drop=True)

feature_cols = [c for c in df.columns if c not in DROP_COLS + ["date_parsed"]]

# Forward-fill per ticker, then median-fill any remaining NaNs
for col in feature_cols:
    df[col] = df.groupby("ticker")[col].transform(lambda x: x.ffill().bfill())
df[feature_cols] = df[feature_cols].fillna(df[feature_cols].median())

X = df[feature_cols].values.astype(np.float32)
y = df[TARGET].values

# Encode labels: -1→0, 0→1, 1→2
le = LabelEncoder()
y_enc = le.fit_transform(y)
class_names = [LABEL_MAP[c] for c in le.classes_]

print(f"Feature matrix : {X.shape[0]} rows × {X.shape[1]} features")
print(f"Classes        : {list(zip(le.classes_, class_names))}")
print(f"Features       : {feature_cols}")

## 3 · Temporal Train / Test Split

> We use a strict chronological 80/20 split to prevent data leakage — future data must never appear in training.

In [ ]:
split_idx = int(len(X) * 0.8)
X_train, X_test = X[:split_idx], X[split_idx:]
y_train, y_test = y_enc[:split_idx], y_enc[split_idx:]

print(f"Train : {len(X_train)} rows  ({df['date_parsed'].iloc[:split_idx].min().date()} → {df['date_parsed'].iloc[:split_idx].max().date()})")
print(f"Test  : {len(X_test)} rows  ({df['date_parsed'].iloc[split_idx:].min().date()} → {df['date_parsed'].iloc[split_idx:].max().date()})")
print("✅ Chronological split — zero data leakage")

## 4 · Model Definitions

In [ ]:
# ── Baseline: Logistic Regression ──────────────────────────────────
baseline_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler",  StandardScaler()),
    ("clf",     LogisticRegression(
        max_iter=1000, class_weight="balanced", C=0.5, random_state=42
    ))
])

# ── Primary: XGBoost (GPU) ──────────────────────────────────────────
primary_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler",  StandardScaler()),
    ("clf",     XGBClassifier(
        n_estimators=300,
        max_depth=5,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        device="cuda",          # GPU acceleration
        eval_metric="mlogloss",
        random_state=42
    ))
])

models = {
    "Logistic_Regression": baseline_pipe,
    "XGBoost_GPU":         primary_pipe,
}

print("Models defined:")
for name in models:
    print(f"  · {name}")

## 5 · TimeSeriesSplit Cross-Validation

Uses `TimeSeriesSplit` — each fold's test set is always **after** its training set in time.

In [ ]:
tscv = TimeSeriesSplit(n_splits=N_SPLITS)
cv_score = {}

for name, pipe in models.items():
    cv_res = cross_validate(
        pipe, X_train, y_train,
        cv=tscv,
        scoring=["accuracy", "f1_macro"],
        return_train_score=True,
        n_jobs=-1
    )
    cv_score[name] = {
        "cv_accuracy_mean":   float(np.mean(cv_res["test_accuracy"])),
        "cv_accuracy_std":    float(np.std(cv_res["test_accuracy"])),
        "cv_f1_macro_mean":   float(np.mean(cv_res["test_f1_macro"])),
        "cv_f1_macro_std":    float(np.std(cv_res["test_f1_macro"])),
        "train_accuracy_mean": float(np.mean(cv_res["train_accuracy"])),
    }
    print(f"{name:30s}  "
          f"CV acc={cv_score[name]['cv_accuracy_mean']:.3f}±{cv_score[name]['cv_accuracy_std']:.3f}  "
          f"F1-macro={cv_score[name]['cv_f1_macro_mean']:.3f}±{cv_score[name]['cv_f1_macro_std']:.3f}")

# Full fit on entire train set
for name, pipe in models.items():
    pipe.fit(X_train, y_train)
    print(f"✅ {name} fitted")

## 6 · Hold-out Evaluation & Confusion Matrices

In [ ]:
run_records = []
fig, axes = plt.subplots(1, len(models), figsize=(5 * len(models), 4))
if len(models) == 1:
    axes = [axes]

for ax, (name, pipe) in zip(axes, models.items()):
    y_pred = pipe.predict(X_test)

    metrics = {
        "model":           name,
        "timestamp":       TIMESTAMP,
        "n_train":         int(len(X_train)),
        "n_test":          int(len(X_test)),
        "accuracy":        float(accuracy_score(y_test, y_pred)),
        "f1_macro":        float(f1_score(y_test, y_pred, average="macro", zero_division=0)),
        "f1_weighted":     float(f1_score(y_test, y_pred, average="weighted", zero_division=0)),
        "precision_macro": float(precision_score(y_test, y_pred, average="macro", zero_division=0)),
        "recall_macro":    float(recall_score(y_test, y_pred, average="macro", zero_division=0)),
        **cv_score[name],
        "per_class": {
            class_names[i]: {
                "precision": float(precision_score(y_test, y_pred, labels=[i], average="macro", zero_division=0)),
                "recall":    float(recall_score(y_test, y_pred, labels=[i], average="macro", zero_division=0)),
                "f1":        float(f1_score(y_test, y_pred, labels=[i], average="macro", zero_division=0)),
            }
            for i in range(len(le.classes_))
        }
    }
    run_records.append(metrics)

    print(f"\n── {name} ──")
    print(f"Accuracy   : {metrics['accuracy']:.3f}")
    print(f"F1-macro   : {metrics['f1_macro']:.3f}")
    print(f"F1-weighted: {metrics['f1_weighted']:.3f}")
    print(classification_report(y_test, y_pred, target_names=class_names, zero_division=0))

    # Confusion matrix
    cm = confusion_matrix(y_test, y_pred, labels=range(len(class_names)))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=class_names, yticklabels=class_names,
                ax=ax, linewidths=0.5)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
    ax.set_title(f"Confusion Matrix\n{name}", fontweight="bold")

    # Serialise model
    pkl_path = MODEL_DIR / f"{name}_{TIMESTAMP}.pkl"
    with open(pkl_path, "wb") as f:
        pickle.dump({"pipeline": pipe, "label_encoder": le, "feature_cols": feature_cols}, f)
    print(f"💾 Saved: {pkl_path.name}")

plt.tight_layout()
cm_path = PLOT_DIR / "confusion_matrices.png"
plt.savefig(cm_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"\n📊 Saved confusion matrices → {cm_path}")

## 7 · Feature Importance & Model Comparison

In [ ]:
xgb_clf = models["XGBoost_GPU"].named_steps["clf"]
importances = xgb_clf.feature_importances_
fi_cols = feature_cols[:len(importances)]

importance_df = (
    pd.DataFrame({"feature": fi_cols, "importance": importances})
    .sort_values("importance", ascending=False)
    .reset_index(drop=True)
)

top_n = min(20, len(importance_df))
fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(
    importance_df["feature"][:top_n][::-1],
    importance_df["importance"][:top_n][::-1],
    color="#2563EB", edgecolor="white", linewidth=0.4
)
ax.set_xlabel("Importance Score", fontsize=11)
ax.set_title("Feature Importance — XGBoost (Top 20)", fontsize=12, fontweight="bold")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
fi_path = PLOT_DIR / "feature_importance_XGBoost.png"
plt.savefig(fi_path, dpi=150)
plt.show()
print(f"📊 Saved → {fi_path}")

In [ ]:
metrics_list = ["accuracy", "f1_macro", "f1_weighted", "precision_macro", "recall_macro"]
compare_df   = pd.DataFrame(run_records).set_index("model")[metrics_list]

fig, ax = plt.subplots(figsize=(9, 4))
x = np.arange(len(metrics_list))
w = 0.35
colors = ["#2563EB", "#F59E0B"]

for i, (model_name, row) in enumerate(compare_df.iterrows()):
    offset = (i - 0.5) * w
    ax.bar(x + offset, row.values, w, label=model_name, color=colors[i], alpha=0.9)

ax.set_xticks(x)
ax.set_xticklabels([m.replace("_", "\n") for m in metrics_list], fontsize=9)
ax.set_ylim(0, 1.15)
ax.set_ylabel("Score")
ax.set_title("Model Comparison — Hold-out Test Set", fontsize=12, fontweight="bold")
ax.legend(fontsize=9)
ax.spines[["top", "right"]].set_visible(False)
for bars_group in ax.containers:
    ax.bar_label(bars_group, fmt="%.2f", fontsize=7, padding=2)
plt.tight_layout()
cmp_path = PLOT_DIR / "model_comparison.png"
plt.savefig(cmp_path, dpi=150)
plt.show()
print(f"📊 Saved → {cmp_path}")

## 8 · SHAP Feature Explainability

SHAP values show **which features actually drove each prediction** — not just global importance. This is the key plot for recruiter portfolios.

In [ ]:
explainer   = shap.TreeExplainer(xgb_clf)
shap_values = explainer.shap_values(X_test)

plt.figure()
shap.summary_plot(
    shap_values, X_test,
    feature_names=feature_cols,
    class_names=class_names,
    show=False
)
shap_path = PLOT_DIR / "shap_summary.png"
plt.savefig(shap_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"📊 Saved → {shap_path}")

## 9 · Champion Model Selection & Run Registry

In [ ]:
# Append to run registry (MLflow-style JSON log)
existing = []
if REGISTRY.exists():
    with open(REGISTRY) as f:
        existing = json.load(f)
existing.extend(run_records)
with open(REGISTRY, "w") as f:
    json.dump(existing, f, indent=2)

# Select champion by F1-macro
champion      = max(run_records, key=lambda r: r["f1_macro"])
champion_src  = MODEL_DIR / f"{champion['model']}_{TIMESTAMP}.pkl"
champion_path = MODEL_DIR / "champion_model.pkl"
shutil.copy(champion_src, champion_path)

print("=" * 50)
print(f"★  Champion : {champion['model']}")
print(f"   Accuracy : {champion['accuracy']:.3f}")
print(f"   F1-macro : {champion['f1_macro']:.3f}")
print(f"   Saved to : {champion_path}")
print("=" * 50)

## 10 · MLflow Experiment Tracking

In [ ]:
mlflow.set_experiment("stock_direction")

with mlflow.start_run(run_name=f"XGBoost_GPU_{TIMESTAMP}"):
    mlflow.log_params(xgb_clf.get_params())
    mlflow.log_metrics({
        "f1_macro":  champion["f1_macro"],
        "accuracy":  champion["accuracy"],
        "f1_weighted": champion["f1_weighted"],
    })
    mlflow.log_artifact(str(champion_path))
    mlflow.log_artifact(str(PLOT_DIR / "shap_summary.png"))
    mlflow.log_artifact(str(PLOT_DIR / "feature_importance_XGBoost.png"))
    mlflow.sklearn.log_model(models["XGBoost_GPU"], "model")

print("✅ MLflow run logged")

## 11 · Summary & Outputs

In [ ]:
summary = {
    "run_timestamp":       TIMESTAMP,
    "champion_model":      champion["model"],
    "champion_f1_macro":   champion["f1_macro"],
    "champion_accuracy":   champion["accuracy"],
    "models_trained":      list(models.keys()),
    "features_used":       feature_cols,
    "n_features":          len(feature_cols),
    "train_size":          int(len(X_train)),
    "test_size":           int(len(X_test)),
}

summary_path = OUT_DIR / "run_summary.json"
with open(summary_path, "w") as f:
    json.dump(summary, f, indent=2)

print("=" * 55)
print("  PHASE 3 COMPLETE")
print("=" * 55)
print(f"  Output dir    : {OUT_DIR}")
print(f"  Champion model: {champion_path}")
print(f"  Run registry  : {REGISTRY}")
print(f"  Run summary   : {summary_path}")
print()

# List all outputs
print("All output files:")
for p in sorted(OUT_DIR.rglob("*")):
    if p.is_file():
        size = p.stat().st_size
        print(f"  {str(p.relative_to(OUT_DIR)):50s}  {size:>8,} bytes")